In [1]:
import requests, json, time, sys

def run_with_timing(prompt: str, model: str = "deepseek-r1:14b"):
    url = "http://localhost:11434/api/generate"
    payload = {"model": model, "prompt": prompt, "stream": True}

    think_tokens, answer_tokens = [], []
    in_think = False
    ttft = None
    t_start = time.perf_counter()

    with requests.post(url, json=payload, stream=True) as resp:
        for line in resp.iter_lines():
            if not line:
                continue
            chunk = json.loads(line)
            tok = chunk.get("response", "")

            if ttft is None and tok:
                ttft = time.perf_counter() - t_start

            # Track which phase we're in
            if "<think>" in tok:
                in_think = True
            if "</think>" in tok:
                in_think = False

            if in_think or "<think>" in tok:
                think_tokens.append(tok)
            else:
                answer_tokens.append(tok)

            sys.stdout.write(tok)
            sys.stdout.flush()

            if chunk.get("done"):
                elapsed = time.perf_counter() - t_start
                total_toks = chunk.get("eval_count", 0)
                prompt_toks = chunk.get("prompt_eval_count", 0)
                gen_toks = total_toks  # eval_count = tokens generated
                
                print(f"\n\n{'─'*50}")
                print(f"Time to first token : {ttft*1000:.0f} ms")
                print(f"Prompt tokens       : {prompt_toks}")
                print(f"Think tokens        : {len(think_tokens)}")
                print(f"Answer tokens       : {len(answer_tokens)}")
                print(f"Total gen time      : {elapsed:.1f} s")
                if elapsed > 0 and gen_toks > 0:
                    print(f"Tokens/sec          : {gen_toks/elapsed:.1f}")
                break

PROMPT = """A bat and a ball cost $1.10 in total.
The bat costs $1.00 more than the ball.
How much does the ball cost? Work through it carefully."""

run_with_timing(PROMPT)

Let's break down the problem step by step.

**Problem Statement:**
A bat and a ball together cost **\$1.10**. The bat costs **\$1.00** more than the ball. We need to find out how much the ball costs.

---

**Step 1:**  
*Define Variables*

- Let \( B \) represent the cost of the ball in dollars.
- Since the bat costs \$1.00 more than the ball, we can express the cost of the bat as:
  
  \[
  \text{Bat} = B + 1
  \]

---

**Step 2:**  
*Set Up the Equation*

The total cost of both the bat and the ball is \$1.10. Therefore:

\[
\text{Ball} + \text{Bat} = 1.10
\]

Substituting the expressions we have:

\[
B + (B + 1) = 1.10
\]

---

**Step 3:**  
*Combine Like Terms*

Simplify the equation by combining like terms:

\[
2B + 1 = 1.10
\]

---

**Step 4:**  
*Solve for \( B \)*

Subtract \$1 from both sides to isolate the term with \( B \):

\[
2B = 1.10 - 1
\]
\[
2B = 0.10
\]

Now, divide both sides by 2:

\[
B = \frac{0.10}{2}
\]
\[
B = 0.05
\]

---

**Conclusion:**

The ball costs **\$0.05

In [2]:
PROMPT = """Write a Python function that finds all prime numbers up to n 
using the Sieve of Eratosthenes. Then explain the time complexity."""

run_with_timing(PROMPT)

To find all prime numbers up to a given number \( n \) efficiently, we can use the Sieve of Eratosthenes algorithm. This method works by iteratively marking the multiples of each prime starting from 2, thus identifying all primes in the range.

**Step-by-Step Explanation:**

1. **Initialization**: Create a boolean array `is_prime` of size \( n+1 \) where each entry is initially set to `True`, except for indices 0 and 1 which are set to `False` since 0 and 1 are not prime numbers.

2. **Iterate Over Potential Primes**: Loop through each number starting from 2 up to the square root of \( n \). For each number \( p \), if it is still marked as prime (`is_prime[p]` is `True`):
   - Mark all multiples of \( p \) (starting from \( p^2 \)) as non-prime by setting their corresponding entries in `is_prime` to `False`.

3. **Collect Primes**: After processing, collect all indices where the value in `is_prime` remains `True`. These indices represent all prime numbers up to \( n \).

**Python Func

In [3]:
# ~/ai-lab/ollama_client.py
import requests
import json
from typing import Iterator

OLLAMA_BASE = "http://localhost:11434"

def stream_generate(
    prompt: str,
    model: str = "deepseek-r1:14b",
    **options,          # temperature, top_p, num_ctx, etc.
) -> Iterator[str]:
    """
    Yields one token string at a time as they arrive.
    The caller decides what to do with them (print, accumulate, parse).
    """
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": True,
        "options": options,
    }

    with requests.post(
        f"{OLLAMA_BASE}/api/generate",
        json=payload,
        stream=True,
        timeout=120,
    ) as resp:
        resp.raise_for_status()
        for raw_line in resp.iter_lines():
            if not raw_line:
                continue
            chunk = json.loads(raw_line)
            yield chunk["response"]          # the token string
            if chunk.get("done"):
                return

In [4]:
# Smoke test — run this first
if __name__ == "__main__":
    for tok in stream_generate("What is 2 + 2?", temperature=0):
        print(tok, end="", flush=True)
    print()

**Solution:**

To find the sum of 2 and 2, follow these simple steps:

1. **Identify the numbers to add:**  
   We have two numbers: 2 and 2.

2. **Add the numbers together:**  
   \[
   2 + 2 = 4
   \]

3. **Final Answer:**  
   The sum of 2 and 2 is \(\boxed{4}\).


In [5]:
from dataclasses import dataclass, field

@dataclass
class ChatSession:
    model: str = "deepseek-r1:14b"
    system: str = ""
    history: list[dict] = field(default_factory=list)
    options: dict = field(default_factory=dict)

    def chat(self, user_message: str) -> str:
        """Send a message, stream the reply, update history, return full reply."""
        self.history.append({"role": "user", "content": user_message})

        messages = []
        if self.system:
            messages.append({"role": "system", "content": self.system})
        messages.extend(self.history)

        payload = {
            "model": self.model,
            "messages": messages,
            "stream": True,
            "options": self.options,
        }

        full_reply = ""
        with requests.post(
            f"{OLLAMA_BASE}/api/chat",
            json=payload,
            stream=True,
            timeout=180,
        ) as resp:
            resp.raise_for_status()
            for raw_line in resp.iter_lines():
                if not raw_line:
                    continue
                chunk = json.loads(raw_line)
                tok = chunk.get("message", {}).get("content", "")
                print(tok, end="", flush=True)
                full_reply += tok
                if chunk.get("done"):
                    break

        print()  # newline after stream ends
        self.history.append({"role": "assistant", "content": full_reply})
        return full_reply

    def reset(self):
        self.history.clear()

In [6]:
session = ChatSession(
    system="You are a terse senior engineer. Give direct answers with no fluff.",
    options={"temperature": 0.3, "num_ctx": 8192},
)

session.chat("What is the KV cache in an LLM?")
session.chat("How does it grow with context length?")
session.chat("What happens on my M4 Pro when it fills the context window?")

The KV cache in large language models (LLMs) is an optimization technique that enhances efficiency by storing intermediate computational results as key-value pairs. Here's a concise breakdown:

1. **Function**: The KV cache stores output states or attention mechanisms from previous layers, allowing the model to retrieve these values quickly instead of recalculating them.

2. **Key Generation**: Keys are typically derived from the layer number and token position, ensuring each cached value is context-specific and avoids conflicts.

3. **Benefits**:
   - **Speed Improvement**: Reduces redundant computations, speeding up processing.
   - **Efficiency**: Lowers memory usage and computational resources, aiding scalability.

4. **Considerations**:
   - **Memory Usage**: Excessive caching can strain memory resources.
   - **Trade-offs**: Caching may not always benefit performance, depending on data reuse frequency.

In essence, the KV cache optimizes LLM operations by efficiently managing int

'When using a large language model (LLM) on an Apple M4 Pro chip, several key interactions occur between the context window and the KV cache:\n\n1. **Context Window**: This refers to the amount of recent data or history that the model can access for processing new inputs. When the context window is full, the model has reached its limit on how much past information it can retain.\n\n2. **KV Cache Interaction**: The KV (Key-Value) cache stores intermediate computations to speed up processing by avoiding redundant calculations. When the context window is full, the model may prioritize newer computations over older ones or clear out cached data to make room for new information.\n\n3. **M4 Pro Handling**: The M4 Pro efficiently manages these processes through optimizations and possibly throttling to ensure smooth operation without overwhelming system resources.\n\n4. **Performance Impact**: Filling the context window might lead to less accurate responses if historical data is limited, but t